##### 🥈 SILVER LAYER
Data Cleaning, Schema Enforcement and Null Handling and Saving as Delta Lake

In [1]:
#Load environment variables from .env files 
from dotenv import load_dotenv
import os

load_dotenv("coding.env")

True

In [2]:
#Initiate Spark Session

import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"
sys.path.append("C:\\hadoop\\bin")

from pyspark.sql import SparkSession


from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("PyProject") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.1.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

In [3]:
#Reading parquet file from BRONZE LAYER to a new dataframe

path = os.getenv("file_path1")
sales = spark.read.parquet(path)


In [4]:
#Call an action to show the Sales data
sales.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|        datetime_now|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|      10|        185|Digital Wallet|  Online|        4/8/2024|            TRUE|2026-03-03 17:55:...|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|            29|       9|        261|Digital Wallet|  Online|       7/23/2023|            TRUE|2026-03-03 17:55:...|
|   TXN_9303719|    CUST_02|            Butchers| Item_12_BUT|          21.5|       2|         43|   Credit Ca

In [5]:
#Check the schema
sales.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Total Spent: string (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: string (nullable = true)
 |-- Discount Applied: string (nullable = true)
 |-- datetime_now: timestamp (nullable = true)



In [6]:
#Defining Schema for SILVER LAYER

from pyspark.sql.types import StructType, StructField,StringType, DateType, TimestampType, FloatType

schema_sales = StructType([StructField("Transaction ID", StringType(), True),
                        StructField("Customer ID", StringType(), True),
                        StructField("Category", StringType(), True),
                        StructField("Item", StringType(), True),
                        StructField("Price Per Unit", FloatType(), True),
                        StructField("Quantity",FloatType(), True),
                        StructField("Total Spent", FloatType(), True),
                        StructField("Payment Method", StringType(), True),
                        StructField("Location", StringType(), True),
                        StructField("Transaction Date",DateType(), True),
                        StructField("Discount Applied", StringType(), True),
                        StructField("datetime_now", TimestampType(), True)
                        ])


In [7]:
#Create Dataframe for SILVER LAYER

from pyspark.sql.functions import col
sales_silver = sales 
for field in schema_sales.fields:
    sales_silver = sales_silver.withColumn(field.name, col(field.name).cast(field.dataType))



In [8]:
from pyspark.sql.functions import expr

sales_silver = sales_silver.withColumn(
    "Transaction Date",
    expr("try_to_timestamp(`Transaction Date`, 'yyyy-MM-dd')")
)

In [9]:
#Check the schema
sales_silver.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: float (nullable = true)
 |-- Quantity: float (nullable = true)
 |-- Total Spent: float (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: timestamp (nullable = true)
 |-- Discount Applied: string (nullable = true)
 |-- datetime_now: timestamp (nullable = true)



Schema Defined

In [10]:
#Call an action to display dataframe

sales_silver.show()

{"ts": "2026-03-03 18:08:51.815", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '4/8/2024' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [8]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o132.showString.\n: org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '4/8/2024' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 6 in cell [8]\n\r\n\tat org.apache.spark.sql.errors.ExecutionError

DateTimeException: [CAST_INVALID_INPUT] The value '4/8/2024' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 6 in cell [8]


Cleaning and Manipulation

In [ ]:
# Cleaning and Manipulation
from pyspark.sql.functions import coalesce, lit, col

#Fix and Transform NULLs

sales_silver_null = sales_silver \
    .withColumn("Transaction ID", coalesce(col("Transaction ID"), lit("Unknown"))) \
    .withColumn("Customer ID", coalesce(col("Customer ID"), lit("Unknown"))) \
    .withColumn("Category", coalesce(col("Category"), lit("Unknown"))) \
    .withColumn("Item", coalesce(col("Item"), lit("Unknown"))) \
    .withColumn("Payment Method", coalesce(col("Payment Method"), lit("Unknown"))) \
    .withColumn("Location", coalesce(col("Location"), lit("Unknown"))) \
    .withColumn("Transaction Date", coalesce(col("Transaction Date"), lit("Unknown"))) \
    .withColumn("Discount Applied", coalesce(col("Discount Applied"), lit("Unknown")))



In [ ]:
#Call an action to show the dataframe
sales_silver_null.show()

{"ts": "2026-03-03 17:57:38.970", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '4/8/2024' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [26]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o397.showString.\n: org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '4/8/2024' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 6 in cell [26]\n\r\n\tat org.apache.spark.sql.errors.ExecutionErr

DateTimeException: [CAST_INVALID_INPUT] The value '4/8/2024' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 6 in cell [26]


In [ ]:
# Removing Duplicates

sales_silver_unique = sales_silver_null.distinct() 


In [ ]:
# Call an action to show the unique rows

sales_silver_unique.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|        datetime_now|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|   TXN_5328604|    CUST_07|                Food|Item_20_FOOD|          33.5|    10.0|      335.0|          Cash|  Online|      2024-07-08|           False|2026-03-03 13:23:...|
|   TXN_2634868|    CUST_13|          Patisserie| Item_11_PAT|          20.0|     4.0|       80.0|          Cash|In-store|      2024-06-26|           False|2026-03-03 13:23:...|
|   TXN_4793201|    CUST_12|           Beverages| Item_16_BEV|          27.5|     2.0|       55.0|   Credit Ca

In [ ]:
# Verify Row counts

print("Before duplicates: ", sales_silver_null.count())
print("After duplicates: ", sales_silver_unique.count())

Before duplicates:  12575
After duplicates:  12575


No Duplicates Found

In [ ]:
#Check the value of particular columns 

In [ ]:
# Rename columns

sales_silver_unique = sales_silver_unique \
    .withColumnRenamed("Transaction ID", "Transaction_ID") \
    .withColumnRenamed("Customer ID", "Customer_ID") \
    .withColumnRenamed("Price Per Unit", "Price_Per_Unit") \
    .withColumnRenamed("Total Spent", "Total_Spent") \
    .withColumnRenamed("Payment Method", "Payment_Method") \
    .withColumnRenamed("Transaction Date", "Transaction_Date") \
    .withColumnRenamed("Discount Applied", "Discount_Applied")

In [ ]:
#Check the value of particular columns 

df__check = sales_silver_unique.withColumn('is_equal', (col('Price_Per_Unit')* col('Quantity')) == col('Total_Spent'))


In [ ]:
df__check.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+--------+
|Transaction_ID|Customer_ID|            Category|        Item|Price_Per_Unit|Quantity|Total_Spent|Payment_Method|Location|Transaction_Date|Discount_Applied|        datetime_now|is_equal|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+--------+
|   TXN_5328604|    CUST_07|                Food|Item_20_FOOD|          33.5|    10.0|      335.0|          Cash|  Online|      2024-07-08|           False|2026-03-03 13:23:...|    true|
|   TXN_2634868|    CUST_13|          Patisserie| Item_11_PAT|          20.0|     4.0|       80.0|          Cash|In-store|      2024-06-26|           False|2026-03-03 13:23:...|    true|
|   TXN_4793201|    CUST_12|           Beverages| Item_16_BEV|   

In [ ]:
df__check.filter(col('is_equal')== False).show()

+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+------------+--------+
|Transaction_ID|Customer_ID|Category|Item|Price_Per_Unit|Quantity|Total_Spent|Payment_Method|Location|Transaction_Date|Discount_Applied|datetime_now|is_equal|
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+------------+--------+
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+------------+--------+



No Mismatches Found

In [ ]:
final_delta_lake = df__check.drop('is_equal')

In [ ]:
final_delta_lake.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|Transaction_ID|Customer_ID|            Category|        Item|Price_Per_Unit|Quantity|Total_Spent|Payment_Method|Location|Transaction_Date|Discount_Applied|        datetime_now|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|   TXN_5328604|    CUST_07|                Food|Item_20_FOOD|          33.5|    10.0|      335.0|          Cash|  Online|      2024-07-08|           False|2026-03-03 13:23:...|
|   TXN_2634868|    CUST_13|          Patisserie| Item_11_PAT|          20.0|     4.0|       80.0|          Cash|In-store|      2024-06-26|           False|2026-03-03 13:23:...|
|   TXN_4793201|    CUST_12|           Beverages| Item_16_BEV|          27.5|     2.0|       55.0|   Credit Ca

In [ ]:
# Save as Delta Lake

silver_path = os.getenv("SILVER_PATH")
delta_lake_path = sales_silver_unique.write.mode("overwrite").format("delta").save(silver_path)

DELTA LAKE CREATED